In [1]:
!pip install -r ~/filestore/shared/requirements.txt
!pip install peft matplotlib scikit-learn
!pip install attention_map_diffusers

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import warnings

warnings.filterwarnings("ignore")

In [3]:
import diffusers # type: ignore
import transformers # type: ignore

print(f'Diffusers version: {diffusers.__version__}')
print(f'Transformers version: {transformers.__version__}')

Diffusers version: 0.31.0
Transformers version: 4.46.3


In [4]:
import gc
import random
import numpy as np # type: ignore
import plotly.express as px# type: ignore
import pandas as pd# type: ignore
from diffusers import AutoencoderKL, StableDiffusionXLPipeline # type: ignore
from PIL import Image # type: ignore
from safetensors.torch import load_file # type: ignore
import torch # type: ignore
from sklearn.preprocessing import StandardScaler# type: ignore
import mplcursors# type: ignore
from torchvision.transforms import v2 # type: ignore
from sklearn.decomposition import PCA# type: ignore
import matplotlib.pyplot as plt# type: ignore
from huggingface_hub import snapshot_download  # type: ignore
import os

def flush():
    gc.collect()
    torch.cuda.empty_cache()

seed = 42
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используемое устройство: {device}")

2026-07-04 18:29:13.577315: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-04 18:29:13.629831: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-04 18:29:14.738478: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Используемое устройство: cuda


In [5]:
#random.seed(seed)
#np.random.seed(seed)
# PyTorch CPU
#torch.manual_seed(seed)
# PyTorch GPU (все доступные GPU)
#torch.cuda.manual_seed(seed)
#torch.cuda.manual_seed_all(seed)  # если несколько GPU
# Делает свёртки и другие операции детерминированными
#torch.backends.cudnn.deterministic = True
#torch.backends.cudnn.benchmark = False

In [6]:
# Пути (вы уже задали)
CACHE_DIR = r'/home/jupyter/filestore/shared'
BASE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"

# 1. Функция загрузки LoRA (без изменений, работает)
def load_lora(pipeline, lora_path: str, lora_scale: float = 0.8, fuse: bool = True):
    """
    Загрузка LoRA адаптера.
    - fuse=True: сливает веса в модель (постоянно) — проще, но нельзя выключить.
    - fuse=False: загружает, но не сливает — тогда при генерации нужно передавать scale через cross_attention_kwargs.
    """
    print(f"Loading LoRA: {lora_path}")
    try:
        pipeline.load_lora_weights(lora_path)
        if fuse:
            pipeline.fuse_lora(lora_scale=lora_scale)
            print(f"LoRA fused with scale {lora_scale}")
        else:
            # В таком случае при вызове pipeline нужно добавить cross_attention_kwargs={"scale": lora_scale}
            print(f"LoRA loaded (not fused). Use cross_attention_kwargs during generation.")
    except Exception as e:
        print(f"Error loading LoRA: {e}")
    return pipeline

# 2. Функция загрузки SDXL без ControlNet
def load_sdxl_pipeline(use_custom_vae: bool = True):
    """
    Загружает базовый SDXL пайплайн без ControlNet и IP-Adapter.
    """
    print("Loading SDXL pipeline...")
    
    vae = None
    if use_custom_vae:
        vae = AutoencoderKL.from_pretrained(
            "madebyollin/sdxl-vae-fp16-fix",
            torch_dtype=torch.float16,
            cache_dir=CACHE_DIR,
        )
        print("Loaded custom VAE")

    pipeline = StableDiffusionXLPipeline.from_pretrained(
        BASE_MODEL,
        vae=vae,
        torch_dtype=torch.float16,
        variant="fp16",
        use_safetensors=True,
        cache_dir=CACHE_DIR,
    )
    pipeline = pipeline.to(device)
    print("Pipeline ready.\n")
    return pipeline

In [7]:
print("Загрузка базового SDXL...")
pipe = load_sdxl_pipeline()

Загрузка базового SDXL...
Loading SDXL pipeline...
Loaded custom VAE


Loading pipeline components...: 100%|██████████| 7/7 [00:06<00:00,  1.03it/s]


Pipeline ready.



In [8]:
os.makedirs("/home/jupyter/project/wo_lora", exist_ok=True)
os.makedirs("/home/jupyter/project/with_lora", exist_ok=True)

In [9]:
prompts = [
    "in the style of kazimir malevich, a highly detailed suprematist composition of an abstract face formed by intersecting black squares, red circles, blue triangles, and yellow rectangles, geometric facial features, bold primary colors on a white canvas, sharp lines, dynamic asymmetry, avant-garde portrait, revolutionary art, digital illustration, 8k",
    "kazimir malevich style, a woman's face broken into floating geometric shards, black and white polygons contrasting with bright red lips and a green eye, the composition feels like a shattered mirror reassembled as suprematist art, metallic textures, dramatic lighting from below, abstract expressionism, detailed digital painting, trending on artstation",
    "kazimir malevich art, an interior scene of a minimalist living room with white walls and a black floor, a large red square painting dominates one wall, geometric furniture: a yellow triangular chair, a blue cylindrical table, a green rectangular sofa, surreal shadows, overhead neon lighting, contemporary digital art, masterpiece",
    "kazimir malevich, a group of five people standing in a line, their bodies reduced to geometric silhouettes of various colors and sizes, one figure is a black rectangle, another a red trapezoid, dynamic diagonal lines cutting through the composition, urban background with floating geometric shapes, suprematist energy, photorealistic digital render",
    "kazimir malevich style, a small bedroom with a bed shaped like a white rectangle, a round black mirror on the wall, a crimson triangle window showing a night sky with abstract stars, a yellow parallelogram rug on the gray floor, soft diffused light, geometric minimalism, highly detailed interior design, 8k, trending on cgsociety",
    "in the style of kazimir malevich, a male profile constructed entirely from overlapping planes of gray, black, and white, a single red semicircle forms the eye, sharp angular jawline, the background is a complex grid of intersecting lines and floating geometric elements, suprematism, futuristic, digital art, hd",
    "kazimir malevich art, a vast geometric landscape with fields divided into colored rectangles, a small village of cubic houses with triangular roofs, tiny human figures walking along a black diagonal path, the sky is a white expanse with a red square sun, early morning light, dreamlike, oil painting texture, digital matte painting",
    "kazimir malevich, a close-up portrait of a female face, the features are fragmented into polygons of white, black, and shades of blue, the lips are a vivid red triangle, the eyes are concentric circles, floating geometric accessories like earrings, abstract expression, highly stylized, digital illustration, sharp focus, intricate",
    "kazimir malevich style, a spacious kitchen with white geometric cabinets, a red triangular table in the center, bar stools shaped like black cylinders, a large window divided into colored rectangular panes, potted plant with round green leaves, natural daylight streaming in, minimalist suprematist interior, photorealistic, 8k",
    "in the style of kazimir malevich, three abstract figures walking through a futuristic city composed entirely of floating black squares, white rectangles, and colored beams, the ground is a checkerboard, each figure is a dynamic arrangement of geometric shapes, motion blur, cinematic lighting, revolutionary art, digital painting",
    "kazimir malevich art, an extreme close-up of a human eye, the iris is a complex pattern of concentric red, blue, and yellow circles, the pupil is a black square, eyelashes made of tiny parallel lines, surrounding skin fragmented into geometric facets, suprematist detail, hyperrealistic, digital macro photography",
    "kazimir malevich, a dynamic arrangement of yellow, green, and black trapezoids hovering in space, the overall silhouette suggests a human face in profile, sharp intersecting lines create a sense of movement, cosmic background with white crosses, revolutionary composition, high contrast, abstract portrait, digital art, trending",
    "kazimir malevich style, a home office with a desk shaped like an elongated black rectangle, a white cylindrical lamp, a red square clock, shelves holding suprematist sculptures, the wallpaper is a grid of tiny geometric patterns, sunlight casting sharp shadows, realistic render, architectural visualization, 8k",
    "in the style of kazimir malevich, a large crowd of people seen from a bird's-eye view, each individual is a tiny colored rectangle or circle moving in different directions, the overall pattern creates a dynamic abstract composition, urban square with geometric patterns on the ground, rhythmic, suprematist, digital illustration",
    "kazimir malevich art, a self-portrait of the artist as a black square wearing a white shirt collar and a bright blue tie, the background is a chaotic mix of floating geometric shapes, ironic and playful, oil on canvas texture, digital recreation, homage to malevich, bold and innovative",
    "kazimir malevich, an abstract bedroom with a bed represented by a large white rectangle, a red triangle blanket, and a black square pillow, a round yellow rug, geometric shadows cast by an unseen window, minimalist and serene, 3d render, architectural interior, photorealistic, soft lighting",
    "kazimir malevich style, a couple embracing transformed into intersecting planes of passionate red and deep black, their bodies are a collage of circles and triangles, white background with delicate black lines indicating motion, suprematist romance, emotional, digital art, expressive, detailed",
    "in the style of kazimir malevich, a child's face constructed from simple colored building blocks, blue square eyes, yellow triangle nose, red rectangle mouth, white background with primary colored shapes floating around, innocent and playful, vector art, digital illustration, bright and cheerful",
    "kazimir malevich art, a fashion boutique window display featuring mannequins wearing geometric dresses, one mannequin is a black square, another a red triangle, the dresses are sharp and avant-garde, lighting creates dramatic shadows, reflective floor, suprematist fashion, commercial photography, highly detailed",
    "kazimir malevich, a complex abstract composition representing a human face, using only black, white, red, blue, and yellow geometric shapes, the eyes are two black squares of different sizes, the mouth is a red trapezoid, surrounded by dynamic diagonal lines and floating circles, suprematism masterpiece, digital art, 8k, trending on artstation"
]

In [10]:
negative_prompt = "low quality, worst quality, jpeg artifacts, blurry, noisy, oversaturated, realistic, photorealistic, 3d render, detailed shading, complex textures, gradient, fine art, baroque, rococo, impressionism, classical, portrait photograph, nude, deformed, ugly, extra limbs, fused fingers, bad anatomy"


for i in range(len(prompts)):
    
    image = pipe(
        prompt=prompts[i],
        negative_prompt=negative_prompt,
        num_inference_steps=30,
        guidance_scale=7,
        height=1024,
        width=1024,
        generator=torch.Generator(device="cuda").manual_seed(42+i)
    ).images[0]

    image.save(f'/home/jupyter/project/wo_lora/{i}.png')

100%|██████████| 30/30 [00:24<00:00,  1.23it/s]


In [12]:
from pathlib import Path

In [13]:
pipe = load_lora(pipe, Path("~/project/diffusionOchka/FF-Style-Kazimir-Malevich-v2.LoRA.safetensors").expanduser(), lora_scale=0.8, fuse=True)

Loading LoRA: /home/jupyter/project/diffusionOchka/FF-Style-Kazimir-Malevich-v2.LoRA.safetensors
LoRA fused with scale 0.8


In [15]:
negative_prompt = "low quality, worst quality, jpeg artifacts, blurry, noisy, oversaturated, realistic, photorealistic, 3d render, detailed shading, complex textures, gradient, fine art, baroque, rococo, impressionism, classical, portrait photograph, nude, deformed, ugly, extra limbs, fused fingers, bad anatomy"


for i in range(len(prompts)):
    
    image = pipe(
        prompt=prompts[i],
        negative_prompt=negative_prompt,
        num_inference_steps=30,
        guidance_scale=7,
        height=1024,
        width=1024,
        generator=torch.Generator(device="cuda").manual_seed(42+i)
    ).images[0]

    image.save(f'/home/jupyter/project/with_lora/{i}.png')

100%|██████████| 30/30 [00:24<00:00,  1.23it/s]


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

# 1. Настройка предобработки картинок
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
imsize = 512

loader = transforms.Compose([
    transforms.Resize((imsize, imsize)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def image_loader(image_name):
    image = Image.open(image_name).convert('RGB')
    image = loader(image).unsqueeze(0)
    return image.to(device, torch.float)

# 2. Функция вычисления Матрицы Грама
def gram_matrix(input_tensor):
    batch_size, channels, height, width = input_tensor.size()
    features = input_tensor.view(batch_size * channels, height * width)
    G = torch.mm(features, features.t())
    # Нормализуем значения, чтобы размер картинки не влиял на результат
    return G.div(batch_size * channels * height * width)

# 3. Загрузка VGG19 для извлечения стиля (используем слои conv1_1, conv2_1, conv3_1, conv4_1)
vgg = models.vgg19(pretrained=True).features.to(device).eval()
style_layers = ['0', '5', '10', '19', '28'] # Индексы слоев свертки VGG19

def calculate_style_loss(img_ref, img_test):
    style_loss = 0.0
    x_ref = img_ref.clone()
    x_test = img_test.clone()
    
    for name, layer in vgg._modules.items():
        x_ref = layer(x_ref)
        x_test = layer(x_test)
        if name in style_layers:
            gm_ref = gram_matrix(x_ref)
            gm_test = gram_matrix(x_test)
            style_loss += nn.functional.mse_loss(gm_ref, gm_test).item()
            
    return style_loss

: 

: 

In [20]:
ref_styles = [image_loader("/home/jupyter/project/diffusionOchka/ref_malevich/ref_malevich_1.jpeg"), image_loader("/home/jupyter/project/diffusionOchka/ref_malevich/ref_malevich_2.jpeg"), image_loader("/home/jupyter/project/diffusionOchka/ref_malevich/ref_malevich_3.jpeg")]

In [25]:
print('Чем меньше, тем ближе к стилю')

hz_kak_nazvat_by_min = []
hz_kak_nazvat_by_mean = []

for i in range(20):
    img_base = image_loader(f"/home/jupyter/project/wo_lora/{i}.png")       # Генерация без LoRA
    img_lora = image_loader(f"/home/jupyter/project/with_lora/{i}.png")       # Генерация с LoRA
    
    loss_base = []
    loss_lora = []
    for ref_style in ref_styles:
        loss_base.append(calculate_style_loss(ref_style, img_base))
        loss_lora.append(calculate_style_loss(ref_style, img_lora))
        
    loss_base_min = min(loss_base)
    loss_lora_min = min(loss_lora)
    
    loss_base_mean = sum(loss_base)/3
    loss_lora_mean = sum(loss_lora)/3
        
    hz_kak_nazvat_by_min.append(((loss_base_min - loss_lora_min) / loss_base_min))
    hz_kak_nazvat_by_mean.append(((loss_base_mean - loss_lora_mean) / loss_base_mean))
    print(f'Картинка: {i}')
    print(f"Улучшения в % (считаем по лучшему):  {((loss_base_min - loss_lora_min) / loss_base_min) * 100:.2f}")
    print(f"Улучшения в % (считаем в среднем):  {((loss_base_mean - loss_lora_mean) / loss_base_mean) * 100:.2f}")

mean_loss_by_min = sum(hz_kak_nazvat_by_min) / len(hz_kak_nazvat_by_min)
mean_loss_by_mean = sum(hz_kak_nazvat_by_mean) / len(hz_kak_nazvat_by_mean)

print(f"Улучшение по всем картинкам в среднем (мо минимальнму лосу): % {mean_loss_by_min * 100:.2f}")
print(f"Улучшение по всем картинкам в среднем (мо среднему лосу): % {mean_loss_by_mean * 100:.2f}")


Чем меньше, тем ближе к стилю
Картинка: 0
Улучшения в % (считаем по лучшему):  0.00
Улучшения в % (считаем в среднем):  0.00
Картинка: 1
Улучшения в % (считаем по лучшему):  0.00
Улучшения в % (считаем в среднем):  0.00
Картинка: 2
Улучшения в % (считаем по лучшему):  -165.29
Улучшения в % (считаем в среднем):  -71.43
Картинка: 3
Улучшения в % (считаем по лучшему):  -75.97
Улучшения в % (считаем в среднем):  -41.86
Картинка: 4
Улучшения в % (считаем по лучшему):  11.50
Улучшения в % (считаем в среднем):  -22.65
Картинка: 5
Улучшения в % (считаем по лучшему):  11.24
Улучшения в % (считаем в среднем):  -4.92
Картинка: 6
Улучшения в % (считаем по лучшему):  92.05
Улучшения в % (считаем в среднем):  88.58
Картинка: 7
Улучшения в % (считаем по лучшему):  74.91
Улучшения в % (считаем в среднем):  65.80
Картинка: 8
Улучшения в % (считаем по лучшему):  20.15
Улучшения в % (считаем в среднем):  18.78
Картинка: 9
Улучшения в % (считаем по лучшему):  94.49
Улучшения в % (считаем в среднем):  93.6